In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import math
import glob
import os

In [ ]:
dir = "/home/tim/cluster/openmpi_wildcard_test/experiment_log"
# Find all CSV files in the directory
csv_files = glob.glob(os.path.join(dir, "*.csv"))
# Read and concatenate
df = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)

#df = pd.read_csv("experiment_log")
df["max_queue"] = df[["pq_max", "uq_max"]].max(axis=1)

df["wildcard_usage"] = df["sequence"].str.contains("with_wildcard")

df.dropna(inplace=True) # the invalid experiments with wildcards but no wildcard support
df

In [ ]:
%%script false --no-raise-error
# Compute max(PRQ_max, UMQ_max)
df["max_queue"] = df[["pq_max", "uq_max"]].max(axis=1)

num_t = 1
# Separate data by wildcard_usage
df_no_wildcard = df[(df["wildcard_usage"] == False) & (df["num_threads"] == num_t)]
df_with_wildcard = df[(df["wildcard_usage"] == True) & (df["num_threads"] == num_t)]

# Plot without wildcard usage
plt.figure()
for impl, group in df_no_wildcard.groupby("implementation"):
    plt.scatter(group["max_queue"], group["ops_per_sec"], label=impl)
plt.ylabel("Operations per Second")
plt.xlabel("Maximum Queue size max(PRQ,UMQ)")
plt.title("Performance vs Queue Size (No Wildcards)")
plt.legend()
plt.grid(True)
plt.show()

# Plot with wildcard usage
plt.figure()
for impl, group in df_with_wildcard.groupby("implementation"):
    plt.scatter(group["max_queue"], group["ops_per_sec"], label=impl)
plt.ylabel("Operations per Second")
plt.xlabel("Maximum Queue size max(PRQ,UMQ)")
plt.title("Performance vs Queue Size (With Wildcards)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Compute max(PRQ_max, UMQ_max)
df["max_queue"] = df[["pq_max", "uq_max"]].max(axis=1)



num_t = 1
# Separate data by wildcard_usage
sequences = df["sequence"].unique()
num_sequences = len(sequences)
# Decide subplot grid (square-ish layout)
ncols = math.ceil(math.sqrt(num_sequences))
nrows = math.ceil(num_sequences / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)

for ax, seq in zip(axes.ravel(), sequences):
    subset = df[df["sequence"] == seq]
    subset = subset[subset["num_threads"] == num_t]
    # Only keep implementations containing "no_wild" or "default"
    subset = subset[
        subset["implementation"].str.contains("no_wild|default", regex=True)
    ]

    # Plot "no_wild" first
    for impl, group in subset.groupby("implementation"):
        if "default" not in impl:
            ax.scatter(group["max_queue"], group["ops_per_sec"], label=impl, alpha=.5)

    # Plot "default" last so it is on top and clearly visible with the edgecolor
    for impl, group in subset.groupby("implementation"):
        if "default" in impl:
            ax.scatter(group["max_queue"], group["ops_per_sec"], label=impl, alpha=.5, edgecolors="black")

    ax.set_ylabel("Operations per Second")
    ax.set_xlabel("Maximum Queue size max(PRQ,UMQ)")
    ax.set_title(f"{seq}")
    ax.grid(True)

# Add legend once (outside subplots)
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(labels))
#plt.title("Performance vs Queue Size")

plt.tight_layout(rect=[0, 0, 1, 0.95])  # leave space for legend
plt.show()



In [ ]:
# Compute max(PRQ_max, UMQ_max)
df["max_queue_avg"] = df[["pq_avg", "uq_avg"]].max(axis=1)



num_t = 1
# Separate data by wildcard_usage
sequences = df["sequence"].unique()
num_sequences = len(sequences)
# Decide subplot grid (square-ish layout)
ncols = math.ceil(math.sqrt(num_sequences))
nrows = math.ceil(num_sequences / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)

for ax, seq in zip(axes.ravel(), sequences):
    subset = df[df["sequence"] == seq]
    subset = subset[subset["num_threads"] == num_t]
    # Only keep implementations containing "no_wild" or "default"
    subset = subset[
        subset["implementation"].str.contains("no_wild|default", regex=True)
    ]

    # Plot "no_wild" first
    for impl, group in subset.groupby("implementation"):
        if "default" not in impl:
            ax.scatter(group["max_queue_avg"], group["ops_per_sec"], label=impl, alpha=.5)

    # Plot "default" last so it is on top and clearly visible with the edgecolor
    for impl, group in subset.groupby("implementation"):
        if "default" in impl:
            ax.scatter(group["max_queue_avg"], group["ops_per_sec"], label=impl, alpha=.5, edgecolors="black")

    ax.set_ylabel("Operations per Second")
    ax.set_xlabel(" Average Queue size")
    ax.set_title(f"{seq}")
    ax.grid(True)

# Add legend once (outside subplots)
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(labels))
#plt.title("Performance vs Queue Size")

plt.tight_layout(rect=[0, 0, 1, 0.95])  # leave space for legend
plt.show()



In [ ]:
%%script false --no-raise-error
# Separate data by wildcard_usage
df_range = df[df["max_queue"].between(0, 200)]
df_no_wildcard = df_range[df_range["wildcard_usage"] == False]
df_with_wildcard = df_range[df_range["wildcard_usage"] == True]

# Plot without wildcard usage
plt.figure()
for impl, group in df_no_wildcard.groupby("implementation"):
    plt.scatter(group["num_threads"], group["ops_per_sec"], label=impl)
plt.ylabel("Operations per Second")
plt.xlabel("Number Of Threads")
plt.title("Performance vs Num threads (No Wildcards)")
plt.legend()
plt.grid(True)
plt.show()

# Plot with wildcard usage
plt.figure()
for impl, group in df_with_wildcard.groupby("implementation"):
    plt.scatter(group["num_threads"], group["ops_per_sec"], label=impl)
plt.ylabel("Operations per Second")
plt.xlabel("Number Of Threads")
plt.title("Performance vs Num threads (No Wildcards)")
plt.legend()
plt.grid(True)
plt.show()